In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Localiza o repositório mesmo quando o notebook é executado a partir de notebooks/.
_inicio = Path.cwd().resolve()
_candidatos_raiz = [_inicio, *_inicio.parents, next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks').is_dir() and (p / 'configs').is_dir())]
ROOT = next((p for p in _candidatos_raiz if (p / 'notebooks').is_dir() and (p / 'results').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do repositório.')

# O primeiro caminho permite incorporar futuramente o DOE ao repositório; os demais
# mantêm compatibilidade com a organização atual dos arquivos da dissertação.
_candidatos_dados = [
    ROOT / 'data' / 'applied' / 'VRF_artigo.xlsx',
    ROOT / 'data' / 'applied' / 'VRF_artigo.xlsx',
    ROOT / 'data' / 'applied' / 'VRF_artigo.xlsx',
]
DATA_FILE = next((p for p in _candidatos_dados if p.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError('VRF_artigo.xlsx não foi encontrado nos caminhos configurados.')

FACTOR_COLS = ['cs', 'f', 'md']
RESPONSE_COLS = ['T', 'MTTF', 'WR', 'Ra', 'Rt', 'Kp', 'ROI', 'OEE']
OUT_DIR = ROOT / 'results' / 'applied' / 'figures_dissertation'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PNG = OUT_DIR / 'figura_01_observados_preditos_rsm_3x3.png'
OUT_PDF = OUT_DIR / 'figura_01_observados_preditos_rsm_3x3.pdf'


In [ ]:
df = pd.read_excel(DATA_FILE)
faltantes = [c for c in FACTOR_COLS + RESPONSE_COLS if c not in df.columns]
if faltantes:
    raise KeyError(f'Colunas ausentes no arquivo de entrada: {faltantes}')

def matriz_quadratica_completa(dados, fatores):
    x = dados[fatores].astype(float)
    partes = [np.ones((len(x), 1)), x.to_numpy()]
    partes.append(np.column_stack([x[c].to_numpy() ** 2 for c in fatores]))
    partes.append(np.column_stack([(x[a] * x[b]).to_numpy() for a, b in combinations(fatores, 2)]))
    return np.column_stack(partes)

X = matriz_quadratica_completa(df, FACTOR_COLS)
Y = df[RESPONSE_COLS].astype(float).to_numpy()
B = np.linalg.pinv(X) @ Y
Y_PRED = X @ B

n, p = X.shape
sse = np.sum((Y - Y_PRED) ** 2, axis=0)
sst = np.sum((Y - Y.mean(axis=0)) ** 2, axis=0)
R2_AJUSTADO = 1.0 - (sse / (n - p)) / (sst / (n - 1))


In [ ]:
CM = 1 / 2.54
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 7.5,
    'axes.titlesize': 8.5,
    'axes.labelsize': 8.0,
    'xtick.labelsize': 6.5,
    'ytick.labelsize': 6.5,
    'axes.linewidth': 0.7,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

fig = plt.figure(figsize=(16 * CM, 13.2 * CM), facecolor='white')
gs = fig.add_gridspec(3, 6, left=0.085, right=0.995, bottom=0.065, top=0.975, wspace=0.70, hspace=0.30)
posicoes = [
    (0, slice(0, 2)), (0, slice(2, 4)), (0, slice(4, 6)),
    (1, slice(0, 2)), (1, slice(2, 4)), (1, slice(4, 6)),
    (2, slice(1, 3)), (2, slice(3, 5)),
]
letras = 'abcdefgh'
cor_pontos = '#4C72B0'

for j, (resposta, posicao) in enumerate(zip(RESPONSE_COLS, posicoes)):
    ax = fig.add_subplot(gs[posicao[0], posicao[1]])
    observado = Y[:, j]
    predito = Y_PRED[:, j]

    minimo = min(observado.min(), predito.min())
    maximo = max(observado.max(), predito.max())
    margem = 0.055 * (maximo - minimo if maximo > minimo else 1.0)
    limites = (minimo - margem, maximo + margem)

    ax.scatter(
        observado, predito, s=20, color=cor_pontos, alpha=0.88,
        edgecolor='white', linewidth=0.35, zorder=3
    )
    ax.plot(limites, limites, color='#333333', linestyle='--', linewidth=0.9, zorder=2)
    ax.set_xlim(limites)
    ax.set_ylim(limites)
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'({letras[j]}) {resposta}', pad=3.0, fontweight='bold')
    ax.text(
        0.04, 0.95, f'$R^2_{{aj}}$ = {R2_AJUSTADO[j]:.3f}'.replace('.', ','),
        transform=ax.transAxes, ha='left', va='top', fontsize=6.6,
        bbox=dict(boxstyle='square,pad=0.18', facecolor='white', edgecolor='#888888', linewidth=0.45)
    )
    ax.grid(True, color='#D9D9D9', linewidth=0.45, alpha=0.65)
    ax.set_axisbelow(True)
    ax.tick_params(length=2.4, width=0.6, pad=1.5)
    for spine in ax.spines.values():
        spine.set_color('#666666')

fig.supxlabel('Valor observado', y=0.008, fontsize=8.5)
fig.supylabel('Valor predito', x=0.008, fontsize=8.5)
fig.savefig(OUT_PNG, dpi=300, facecolor='white', bbox_inches='tight', pad_inches=0.02)
fig.savefig(OUT_PDF, facecolor='white', bbox_inches='tight', pad_inches=0.02)
plt.show()
